# EML-k Final Resolution (Session 42)

**Goal**: Definitively resolve the INCONCLUSIVE cases from Session 38–41:
- `|sin(x)|` — piecewise smooth, N=3 singularity, expected separation plateau
- `|cos(x)|` — same barrier, shifted by π/2
- `x*y` — bivariate product, confirms rank-1 tensor completion

**Method**: SVD projection onto the full EML atom span (no greedy/sparse selection).
This gives the exact (dense) approximation floor — non-increasing in N by construction.

**Expected outcomes**:
- `|sin(x)|`, `|cos(x)|`: plateau ~1e-4 → SEPARATION (structural barrier at N=3)
- `x*y`: MSE → 0 at N=4 → DENSE (rank-1 product atoms suffice)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path

from monogate.frontiers.eml_k_resolution import (
    resolve_univariate,
    resolve_bivariate,
    run_all,
)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print('Imports OK')

## Cell 1: Run Resolution for |sin(x)| and |cos(x)|

In [ ]:
# --- |sin(x)| and |cos(x)| univariate resolution ---
MAX_N = 5

res_sin_abs = resolve_univariate(
    'sin_abs', lambda x: np.abs(np.sin(x)), max_n=MAX_N
)
res_cos_abs = resolve_univariate(
    'cos_abs', lambda x: np.abs(np.cos(x)), max_n=MAX_N
)

for res in [res_sin_abs, res_cos_abs]:
    print(f'\n{res.target}: {res.verdict}')
    print(f'  Notes: {res.notes}')
    for c in res.convergence:
        print(f"  N={c['n']}: {c.get('n_independent',0):4d} atoms  MSE={c['mse']:.3e}")

## Cell 2: Convergence Curves — EML-only vs N=3 Singularity Highlight

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, res, color in zip(axes, [res_sin_abs, res_cos_abs], ['steelblue', 'darkorange']):
    ns = [c['n'] for c in res.convergence]
    mses = [c['mse'] for c in res.convergence]
    
    ax.semilogy(ns, mses, 'o-', color=color, linewidth=2, markersize=7, label='SVD floor')
    
    # Highlight N=3 — the Schanuel/singularity transition
    n3_idx = next((i for i, c in enumerate(res.convergence) if c['n'] == 3), None)
    if n3_idx is not None:
        ax.axvline(x=3, color='red', linestyle='--', alpha=0.6, label='N=3 singularity jump')
        ax.scatter([3], [mses[n3_idx]], s=120, zorder=5, color='red')
    
    ax.axhline(y=1e-6, color='green', linestyle=':', alpha=0.8, label='Dense threshold (1e-6)')
    ax.set_title(f'|{res.target.replace("_abs","(x)")}|  [{res.verdict}]')
    ax.set_xlabel('EML depth N')
    ax.set_ylabel('MSE (log scale)')
    ax.legend(fontsize=9)
    ax.set_xticks(ns)
    ax.grid(True, alpha=0.3)

plt.suptitle('EML-k Resolution: |sin| and |cos| Convergence Floors', fontweight='bold')
plt.tight_layout()
Path('../results').mkdir(exist_ok=True)
plt.savefig('../results/eml_k_resolution_convergence.png', bbox_inches='tight')
plt.show()
print('Saved eml_k_resolution_convergence.png')

## Cell 3: Bivariate x*y — Rank-1 Tensor Completion

In [ ]:
res_biv = resolve_bivariate(
    'xy',
    lambda x1, x2: x1 * x2,
    max_n=MAX_N,
)

print(f'x*y bivariate: {res_biv.verdict}')
print(f'  Atoms: {res_biv.n_atoms_1d} (1D) -> {res_biv.n_atoms_biv} (product dict)')
print(f'  Rank: {res_biv.rank}')
print(f'  MSE:  {res_biv.mse_svd:.3e}')
print(f'  Notes: {res_biv.notes}')

# Visualise residual surface
n_vis = 50
xs_vis = np.linspace(-2, 2, n_vis)
X1, X2 = np.meshgrid(xs_vis, xs_vis)
fig = plt.figure(figsize=(12, 4))

ax1 = fig.add_subplot(131, projection='3d')
ax1.plot_surface(X1, X2, X1 * X2, cmap='viridis', alpha=0.9)
ax1.set_title('Target: x·y')

ax2 = fig.add_subplot(132)
ax2.bar(['1D atoms', 'Product atoms', 'Rank'], 
        [res_biv.n_atoms_1d, res_biv.n_atoms_biv, res_biv.rank],
        color=['steelblue', 'darkorange', 'green'])
ax2.set_title('Dictionary size vs rank')
ax2.set_ylabel('Count')

ax3 = fig.add_subplot(133)
ax3.text(0.5, 0.5,
    f'MSE = {res_biv.mse_svd:.2e}\n\nVerdict: {res_biv.verdict}',
    ha='center', va='center', fontsize=14, fontweight='bold',
    color='green' if res_biv.verdict == 'DENSE' else 'red',
    transform=ax3.transAxes)
ax3.axis('off')
ax3.set_title('x*y Resolution')

plt.tight_layout()
plt.savefig('../results/eml_k_bivariate_xy.png', bbox_inches='tight')
plt.show()
print('Saved eml_k_bivariate_xy.png')

## Cell 4: Paper Table — Updated EML-k Classification

In [ ]:
# Load prior results for complete table
prior_results_path = Path('../results/universal_approx_survey.json')
prior = {}
if prior_results_path.exists():
    with open(prior_results_path) as f:
        prior = json.load(f)

# Compile full classification table
table_rows = [
    # (function, EML-k class, N* (min depth for MSE<1e-6), MSE@N*, notes)
    ('exp(x)',      'EML-1',  1, '< 1e-15', 'Exact: 1 node'),
    ('exp(-x)',     'EML-1',  1, '< 1e-15', 'DEML-1 exact'),
    ('1/x',         'EML-2',  2, '< 1e-14', 'eml(0,x) exact'),
    ('x²',          'EML-3',  3, '~2.8e-15', '23-atom SVD witness (Session 41)'),
    ('x³',          'EML-3',  3, '~1.3e-13', 'SVD witness'),
    ('ln(x)',        'EML-3',  3, '< 1e-14', '3-node exact construction'),
    ('√x',          'EML-3',  3, '< 1e-12', 'via exp(ln(x)/2)'),
    ('|sin(x)|',    'EML-∞*', 5, f'~{res_sin_abs.mse_svd:.1e}', f'{res_sin_abs.verdict}: structural plateau'),
    ('|cos(x)|',    'EML-∞*', 5, f'~{res_cos_abs.mse_svd:.1e}', f'{res_cos_abs.verdict}: structural plateau'),
    ('sin(x)',       'EML-∞',  '∞', 'N/A',  'Infinite zeros barrier (proven)'),
    ('cos(x)',       'EML-∞',  '∞', 'N/A',  'Infinite zeros barrier (proven)'),
    ('x·y (biv.)',  f'EML-≤{MAX_N}', MAX_N, f'{res_biv.mse_svd:.1e}', f'{res_biv.verdict}: rank-1 tensor atoms'),
]

header = f"{'Function':<16} {'Class':<10} {'N*':>4} {'MSE@N*':<12} Notes"
print(header)
print('-' * 80)
for fn, cls, n_star, mse, note in table_rows:
    print(f"{fn:<16} {cls:<10} {str(n_star):>4} {mse:<12} {note}")

print()
print('* EML-∞* = EML-∞ approximand (infinite zeros class), but piecewise smooth')
print('  so MSE decays slowly (SEPARATION) rather than proving barrier.')

## Cell 5: Save Full Results for Paper

In [ ]:
full_results = {
    'session': 42,
    'sin_abs': res_sin_abs.to_dict(),
    'cos_abs': res_cos_abs.to_dict(),
    'xy_bivariate': res_biv.to_dict(),
    'table': [
        {'fn': fn, 'class': cls, 'n_star': n_star, 'mse': mse, 'notes': note}
        for fn, cls, n_star, mse, note in table_rows
    ]
}

out = Path('../results/eml_k_resolution_s42.json')
with open(out, 'w', encoding='utf-8') as f:
    json.dump(full_results, f, indent=2)
print(f'Results saved to {out}')

print('\n=== SESSION 42 SUMMARY ===')
print(f'|sin(x)|: {res_sin_abs.verdict}  MSE={res_sin_abs.mse_svd:.2e}')
print(f'|cos(x)|: {res_cos_abs.verdict}  MSE={res_cos_abs.mse_svd:.2e}')
print(f'x*y biv:  {res_biv.verdict}      MSE={res_biv.mse_svd:.2e}')